# SciFact BGE-base embeddings

This notebook is intended for a Colab T4 runtime. It does not connect to the project database. The local script passes stored chunk text directly to `SentenceTransformer.encode` with `normalize_embeddings=False`; this notebook does exactly the same (no prefix, trimming, or other text normalization).

In [ ]:
!pip install -q sentence-transformers==3.4.1 numpy==2.1.3 pandas==2.2.3 pyarrow==18.0.0

In [ ]:
from google.colab import files
import hashlib, json, os, time
import numpy as np
import pandas as pd
import torch
import sentence_transformers
from sentence_transformers import SentenceTransformer

uploaded = files.upload()
parquet_name = next((name for name in uploaded if name.endswith('.parquet')), None)
jsonl_name = next((name for name in uploaded if name.endswith('.jsonl')), None)
sidecar_name = next((name for name in uploaded if name.endswith('.json')), None)
input_name = parquet_name or jsonl_name
if input_name is None or sidecar_name is None:
    raise ValueError('Upload the parquet/JSONL export and its .json sidecar')
sidecar = json.load(open(sidecar_name, encoding='utf-8'))
chunks = pd.read_parquet(input_name, columns=['id', 'text']) if parquet_name else pd.read_json(input_name, lines=True)[['id', 'text']]
if len(chunks) != sidecar['row_count']:
    raise ValueError(f"Row count mismatch: {len(chunks)} != {sidecar['row_count']}")
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()
if sha256_file(input_name) != sidecar['file_sha256']:
    raise ValueError('Export file checksum mismatch')
rows = sorted([(int(row.id), row.text) for row in chunks.itertuples(index=False)], key=lambda row: row[0])
digest = hashlib.sha256()
for chunk_id, text in rows:
    digest.update(f'{chunk_id}\t{text}\n'.encode('utf-8'))
if digest.hexdigest() != sidecar['id_text_sha256']:
    raise ValueError('ID/text checksum mismatch')
print(f'Verified {len(rows)} chunks; model={sidecar["model"]}, dimension={sidecar["dimension"]}')

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError('A CUDA GPU is required; select a Colab T4 runtime')
device = 'cuda'
print(torch.cuda.get_device_name(0))
model = SentenceTransformer('BAAI/bge-base-en-v1.5', device=device)
model.max_seq_length = 512
tokenizer = model.tokenizer
over_512 = sum(len(tokenizer(text, add_special_tokens=True, truncation=False)['input_ids']) > 512 for _, text in rows)
print(f'Chunks over 512 tokens: {over_512}')

In [ ]:
BATCH_SIZE = 64
SEGMENT_SIZE = 1000
segment_dir = 'embedding_segments'
os.makedirs(segment_dir, exist_ok=True)
started = time.monotonic()
for start in range(0, len(rows), SEGMENT_SIZE):
    end = min(start + SEGMENT_SIZE, len(rows))
    embedding_path = os.path.join(segment_dir, f'embeddings_{start:08d}_{end:08d}.npy')
    ids_path = os.path.join(segment_dir, f'ids_{start:08d}_{end:08d}.npy')
    if os.path.exists(embedding_path) and os.path.exists(ids_path):
        print(f'Skipping existing segment {start}:{end}')
        continue
    texts = [text for _, text in rows[start:end]]
    vectors = model.encode(texts, batch_size=BATCH_SIZE, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=False)
    vectors = np.asarray(vectors, dtype=np.float32)
    if vectors.shape != (end - start, 768):
        raise ValueError(f'Unexpected segment shape: {vectors.shape}')
    np.save(embedding_path, vectors)
    np.save(ids_path, np.asarray([chunk_id for chunk_id, _ in rows[start:end]], dtype=np.int64))
    print(f'Saved {start}:{end}')
print(f'Elapsed seconds: {time.monotonic() - started:.2f}')

In [ ]:
embedding_parts = sorted([os.path.join(segment_dir, name) for name in os.listdir(segment_dir) if name.startswith('embeddings_')])
id_parts = sorted([os.path.join(segment_dir, name) for name in os.listdir(segment_dir) if name.startswith('ids_')])
embeddings = np.concatenate([np.load(path) for path in embedding_parts]).astype(np.float32, copy=False)
ids = np.concatenate([np.load(path) for path in id_parts]).astype(np.int64, copy=False)
if embeddings.shape != (len(rows), 768) or ids.shape != (len(rows),):
    raise ValueError(f'Unexpected merged shapes: {embeddings.shape}, {ids.shape}')
np.save('embeddings.npy', embeddings)
np.save('ids.npy', ids)
meta = {
    'model': 'BAAI/bge-base-en-v1.5', 'dimension': 768, 'device': device,
    'dtype': 'float32', 'batch_size': BATCH_SIZE, 'chunks_over_512_tokens': over_512,
    'elapsed_seconds': time.monotonic() - started,
    'torch_version': torch.__version__, 'sentence_transformers_version': sentence_transformers.__version__,
    'count': len(ids)
}
json.dump(meta, open('embeddings_meta.json', 'w', encoding='utf-8'), indent=2)
print(meta)

In [ ]:
files.download('embeddings.npy')
files.download('ids.npy')
files.download('embeddings_meta.json')